In [ ]:
import json
import re
from pathlib import Path

# === TOKEN COUNTER ===
def estimate_tokens(text: str) -> int:
    if not isinstance(text, str):
        return 0
    return len(re.findall(r"\w+|[^\w\s]", text))

# === LOAD FILE ===
json_path = Path("/home/jesse/Desktop/ai/notebooks/memory/chat/conversations.json")
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"🧠 Total conversation threads: {len(data)}")

# === EXTRACT THREAD DATA ===
threads_info = []

for thread in data:
    thread_id = thread.get("id")
    title = thread.get("title", "")
    mapping = thread.get("mapping", {})
    messages = []

    for node in mapping.values():
        msg = node.get("message")
        if not isinstance(msg, dict):
            continue
        content = msg.get("content", {})
        if not isinstance(content, dict):
            continue
        parts = content.get("parts", [])
        if parts and isinstance(parts[0], str):
            messages.append(parts[0])

    token_counts = [estimate_tokens(m) for m in messages]
    token_total = sum(token_counts)
    token_avg = int(token_total / len(token_counts)) if token_counts else 0

    threads_info.append({
        "id": thread_id,
        "title": title,
        "message_count": len(messages),
        "token_total": token_total,
        "token_avg": token_avg,
        "first_msg": messages[0][:100] + "..." if messages else "",
        "longest_msg": (max(messages, key=len)[:100] + "...") if messages else "",
    })

# === SORT & DISPLAY ===
threads_info.sort(key=lambda x: x["token_total"], reverse=True)

for t in threads_info[:5]:
    print(f"\n--- Thread ---")
    print(f"ID: {t['id']}")
    print(f"Title: {t['title']}")
    print(f"Message count: {t['message_count']}")
    print(f"Token count: {t['token_total']}")
    print(f"Avg tokens/message: {t['token_avg']}")
    print(f"First message preview: {t['first_msg']}")
    print(f"Longest message preview: {t['longest_msg']}")